# LeetCode 1943 — Describe the Painting bằng Python

Link đề: https://leetcode.com/problems/describe-the-painting/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Sorting, Prefix Sum (Sweep Line)

Skeleton LeetCode:

```python
class Solution:
    def splitPainting(self, segments: List[List[int]]) -> List[List[int]]:
        ...
```

Lộ trình: **tô từng đơn vị độ dài, lưu tập màu, gộp các ô liền kề có cùng tập** (vét cạn) → **mảng hiệu trên trục số + đánh dấu mọi điểm biên** `O(n + MAX)` → **quét sự kiện trên các điểm biên đã sắp xếp** `O(n log n)` (lời giải nộp) → **`itertools.accumulate` + `pairwise`** (viết gọn).

## 1. Mục tiêu học

1. Biểu diễn bài "nhiều đoạn chồng nhau" bằng **sự kiện**: tại `start` cộng màu, tại `end` trừ màu (**mảng hiệu**).
2. Hiểu vì sao phải **cắt tại mọi điểm biên** (nơi một đoạn bắt đầu hoặc kết thúc), kể cả khi **tổng màu không đổi** — vì đề yêu cầu tách theo **tập màu**, không theo tổng.
3. Quét các điểm biên **đã sắp xếp**, duy trì tổng hiện tại, xuất đoạn `[x_i, x_{i+1})` khi tổng khác 0.
4. Nhận ra màu là **khác nhau** ⇒ hai tập màu bằng nhau ⇔ không có điểm biên ở giữa — nên "cắt tại mọi điểm biên" là **vừa đủ** (số đoạn tối thiểu).
5. Kiểm chứng bằng vét cạn theo từng đơn vị trên dữ liệu ngẫu nhiên có seed (đáp án có nhiều thứ tự ⇒ so sau khi sắp xếp).

## 2. Đề bài & ví dụ

Một bức tranh dài, mảnh biểu diễn bằng trục số, được sơn bởi nhiều đoạn **chồng nhau**; mỗi đoạn một **màu khác nhau**. `segments[i] = [start_i, end_i, color_i]` là đoạn **nửa mở** `[start_i, end_i)` sơn màu `color_i`.

Ở chỗ chồng nhau, các màu **trộn** thành một màu mới — biểu diễn bằng **tập** các màu. Để đơn giản chỉ xuất **tổng** các phần tử của tập.

Hãy mô tả bức tranh bằng **ít nhất** các đoạn nửa mở **không chồng nhau** `[left_j, right_j, mix_j]` (đoạn `[left, right)` có tập màu với tổng `mix`). Bỏ qua phần **không** được sơn. Thứ tự các đoạn tùy ý.

Ví dụ `segments = [[1,4,5],[1,7,7]]` → `[[1,4,12],[4,7,7]]`.

**Ví dụ 1:** `[[1,4,5],[4,7,7],[1,7,9]]` → `[[1,4,14],[4,7,16]]`.

**Ví dụ 2:** `[[1,7,9],[6,8,15],[8,10,7]]` → `[[1,6,9],[6,7,24],[7,8,15],[8,10,7]]`.

**Ví dụ 3:** `[[1,4,5],[1,4,7],[4,7,1],[4,7,11]]` → `[[1,4,12],[4,7,12]]` — **không** được gộp thành `[1,7,12]` vì hai tập màu `{5,7}` và `{1,11}` **khác nhau** (dù cùng tổng 12).

**Ràng buộc:** `1 ≤ segments.length ≤ 2·10⁴`, `1 ≤ start_i < end_i ≤ 10⁵`, `1 ≤ color_i ≤ 10⁹`, các màu đôi một khác nhau.

In [1]:
import random
import time
from collections import defaultdict
from itertools import accumulate, pairwise
from typing import List

examples = [
    ([[1, 4, 5], [4, 7, 7], [1, 7, 9]], [[1, 4, 14], [4, 7, 16]]),
    ([[1, 7, 9], [6, 8, 15], [8, 10, 7]], [[1, 6, 9], [6, 7, 24], [7, 8, 15], [8, 10, 7]]),
    ([[1, 4, 5], [1, 4, 7], [4, 7, 1], [4, 7, 11]], [[1, 4, 12], [4, 7, 12]]),
]

def norm(ans):
    return sorted(map(list, ans))      # đáp án có thể theo thứ tự bất kỳ

for segs, want in examples:
    print(segs, "->", want)

[[1, 4, 5], [4, 7, 7], [1, 7, 9]] -> [[1, 4, 14], [4, 7, 16]]
[[1, 7, 9], [6, 8, 15], [8, 10, 7]] -> [[1, 6, 9], [6, 7, 24], [7, 8, 15], [8, 10, 7]]
[[1, 4, 5], [1, 4, 7], [4, 7, 1], [4, 7, 11]] -> [[1, 4, 12], [4, 7, 12]]


## 3. Kiến thức nền

### 3.1 Đoạn nửa mở `[a, b)`

Gồm các điểm `x` với `a ≤ x < b`. Hai đoạn `[1,4)` và `[4,7)` **không** chồng nhau — chạm nhau tại 4 nhưng 4 chỉ thuộc đoạn sau. Với số nguyên, `[a, b)` gồm các "ô đơn vị" `[a,a+1), …, [b−1, b)`.

In [2]:
a, b = 1, 4
print("các ô đơn vị của [1, 4):", [(x, x + 1) for x in range(a, b)])

các ô đơn vị của [1, 4): [(1, 2), (2, 3), (3, 4)]


### 3.2 Mảng hiệu (difference array)

Cộng `c` vào mọi vị trí trong `[a, b)`: `diff[a] += c`, `diff[b] -= c`; tổng tiền tố của `diff` cho giá trị tại từng vị trí.

In [3]:
diff = [0] * 10
for a, b, c in [[1, 4, 5], [4, 7, 7], [1, 7, 9]]:
    diff[a] += c; diff[b] -= c
print("diff  :", diff)
print("giá trị:", list(accumulate(diff)))

diff  : [0, 14, 0, 0, 2, 0, 0, -16, 0, 0]
giá trị: [0, 14, 14, 14, 16, 16, 16, 0, 0, 0]


### 3.3 `itertools.pairwise`

`pairwise([x0, x1, x2])` → `(x0, x1), (x1, x2)` — tiện để duyệt các đoạn giữa hai điểm biên liên tiếp (Python ≥ 3.10).

In [4]:
print(list(pairwise([1, 4, 7])))

[(1, 4), (4, 7)]


### 3.4 `defaultdict(int)` cho sự kiện thưa

Thay vì mảng dài `10⁵`, chỉ lưu các điểm có sự kiện: `d[start] += c`, `d[end] -= c`, rồi duyệt `sorted(d)`.

In [5]:
d = defaultdict(int)
for a, b, c in [[1, 7, 9], [6, 8, 15], [8, 10, 7]]:
    d[a] += c; d[b] -= c
print(sorted(d.items()))

[(1, 9), (6, 15), (7, -9), (8, -8), (10, -7)]


### 3.5 Vì sao cắt theo **điểm biên**, không theo **thay đổi tổng**

Tại điểm 4 của Ví dụ 3: hai đoạn kết thúc (`−5 −7`) và hai đoạn bắt đầu (`+1 +11`) ⇒ hiệu tại 4 là `0`, tổng **không đổi** (12 → 12), nhưng **tập màu đổi** (`{5,7}` → `{1,11}`) ⇒ vẫn phải cắt. Vì vậy phải ghi nhận **mọi** điểm có sự kiện, kể cả khi `diff = 0`.

In [6]:
d = defaultdict(int)
for a, b, c in examples[2][0]:
    d[a] += c; d[b] -= c
print(dict(d), "| hiệu tại 4 =", d[4], "nhưng 4 vẫn là điểm biên")

{1: 12, 4: 0, 7: -12} | hiệu tại 4 = 0 nhưng 4 vẫn là điểm biên


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2 trên trục số:

```
1        6  7  8     10
|--- 9 --------|               [1,7) màu 9
         |--15----|            [6,8) màu 15
                  |--- 7 --|   [8,10) màu 7
```

Điểm biên: `1, 6, 7, 8, 10`. Giữa hai điểm biên liên tiếp, **không đoạn nào bắt đầu hay kết thúc** ⇒ tập màu không đổi. Tổng trên từng khoảng:

| khoảng | tập màu | tổng |
|---|---|---|
| `[1,6)` | {9} | 9 |
| `[6,7)` | {9, 15} | 24 |
| `[7,8)` | {15} | 15 |
| `[8,10)` | {7} | 7 |

✔ Và vì màu **khác nhau**, tập màu ở hai khoảng kề nhau luôn **khác** (tại điểm biên có ít nhất một màu vào/ra) ⇒ không thể gộp ⇒ đây là mô tả **ít đoạn nhất**.

## 5. Cách 1 — Brute Force / baseline: tô từng ô đơn vị, gộp ô liền kề cùng tập màu

**Ý tưởng:** với mỗi ô `[x, x+1)` lưu **tập** màu phủ nó (frozenset). Duyệt từ trái sang phải, gộp các ô liền nhau có **cùng tập** (khác rỗng) thành một đoạn. Theo đúng định nghĩa ⇒ chuẩn đối chiếu.

In [7]:
def split_brute(segments):
    hi = max(e for _, e, _ in segments)
    cover = [set() for _ in range(hi + 1)]
    for s, e, c in segments:
        for x in range(s, e):
            cover[x].add(c)
    out = []
    x = 1
    while x < hi:
        if not cover[x]:
            x += 1; continue
        y = x
        while y < hi and cover[y] == cover[x]:
            y += 1
        out.append([x, y, sum(cover[x])])
        x = y
    return out

for segs, want in examples:
    print(norm(split_brute(segs)) == norm(want), split_brute(segs))

True [[1, 4, 14], [4, 7, 16]]
True [[1, 6, 9], [6, 7, 24], [7, 8, 15], [8, 10, 7]]
True [[1, 4, 12], [4, 7, 12]]


**Độ phức tạp:** tô `Σ (end − start)` ô — tới `2·10⁴ × 10⁵ = 2·10⁹` thao tác thêm vào tập ⇒ quá chậm, và bộ nhớ lưu tập màu từng ô. **Nút thắt:** tập màu chỉ đổi ở **điểm biên**; giữa chúng không cần xét từng ô.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (sự kiện):** tập màu tại `x` chỉ thay đổi khi `x` là `start` hoặc `end` của một đoạn. Ghi `delta[start] += c`, `delta[end] −= c` và đánh dấu cả hai là **điểm biên**.

**Insight 2 (quét):** duyệt các điểm biên tăng dần `p₀ < p₁ < …`; giữ `cur` = tổng màu trên `[p_i, p_{i+1})` = `Σ_{j ≤ i} delta[p_j]`. *Nếu* `cur > 0` *thì* xuất `[p_i, p_{i+1}, cur]`.

**Insight 3 (tối thiểu):** vì màu **khác nhau** và dương, hai khoảng liền kề có tập màu khác nhau ⇒ không gộp được ⇒ số đoạn xuất ra là tối thiểu. Nhưng **không** được dùng điều kiện "tổng khác nhau" để cắt (Ví dụ 3).

**Lưu ý:** `cur > 0` ⇔ có ít nhất một màu (vì mọi màu `≥ 1`) — dùng để bỏ phần không được sơn.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng hiệu dài `MAX + 1` + mảng đánh dấu điểm biên (`O(n + MAX)`)

In [8]:
def split_array(segments, MAX=10**5):
    diff = [0] * (MAX + 2)
    is_edge = [False] * (MAX + 2)
    for s, e, c in segments:
        diff[s] += c; diff[e] -= c
        is_edge[s] = is_edge[e] = True
    out = []
    cur = 0
    start = None
    for x in range(1, MAX + 1):
        if is_edge[x]:
            if start is not None and cur > 0:
                out.append([start, x, cur])
            cur += diff[x]
            start = x
    return out

for segs, want in examples:
    print(norm(split_array(segs)) == norm(want))

True
True
True


**Độ phức tạp:** `O(n + MAX)` với `MAX = 10⁵` — nhanh, nhưng duyệt cả các vị trí không có sự kiện; phụ thuộc giá trị tọa độ lớn nhất.

### 7.2 Cách 3 — Sự kiện thưa + sắp xếp điểm biên (`O(n log n)`) — lời giải nộp

In [9]:
def split_sweep_demo(segments):
    delta = defaultdict(int)
    for s, e, c in segments:
        delta[s] += c; delta[e] -= c
    pts = sorted(delta)
    out, cur = [], 0
    for a, b in pairwise(pts):
        cur += delta[a]
        print(f"[{a}, {b}): cộng delta[{a}] = {delta[a]:>4} -> tổng {cur}{'  (không sơn, bỏ)' if cur == 0 else ''}")
        if cur:
            out.append([a, b, cur])
    return out

print("=>", split_sweep_demo(examples[1][0]))

[1, 6): cộng delta[1] =    9 -> tổng 9
[6, 7): cộng delta[6] =   15 -> tổng 24
[7, 8): cộng delta[7] =   -9 -> tổng 15
[8, 10): cộng delta[8] =   -8 -> tổng 7
=> [[1, 6, 9], [6, 7, 24], [7, 8, 15], [8, 10, 7]]


In [10]:
class SolutionSweep:
    def splitPainting(self, segments: List[List[int]]) -> List[List[int]]:
        delta = defaultdict(int)
        for s, e, c in segments:
            delta[s] += c
            delta[e] -= c
        out, cur = [], 0
        pts = sorted(delta)
        for i in range(len(pts) - 1):
            cur += delta[pts[i]]
            if cur:
                out.append([pts[i], pts[i + 1], cur])
        return out

for segs, want in examples:
    print(norm(SolutionSweep().splitPainting(segs)) == norm(want))

True
True
True


Điểm then chốt: `delta[4] = 0` ở Ví dụ 3 **vẫn** là một khóa trong `dict` (vì ta đã truy cập nó) ⇒ 4 vẫn là điểm biên. **Độ phức tạp:** `O(n log n)` sắp xếp tối đa `2n` điểm; bộ nhớ `O(n)`.

### 7.3 Cách 4 — `accumulate` + `pairwise`, viết gọn

In [11]:
class SolutionCompact:
    def splitPainting(self, segments: List[List[int]]) -> List[List[int]]:
        delta = defaultdict(int)
        for s, e, c in segments:
            delta[s] += c; delta[e] -= c
        pts = sorted(delta)
        sums = accumulate(delta[p] for p in pts)
        return [[a, b, s] for (a, b), s in zip(pairwise(pts), sums) if s]

for segs, want in examples:
    print(norm(SolutionCompact().splitPainting(segs)) == norm(want))

True
True
True


`zip` dừng ở dãy ngắn hơn (`pairwise` có `len(pts) − 1` phần tử) nên phần tử `accumulate` cuối (luôn bằng 0) bị bỏ qua tự nhiên.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 3:

| điểm `p` | `delta[p]` | khoảng `[p, p_next)` | `cur` | xuất? |
|---|---|---|---|---|
| 1 | +5 +7 = 12 | `[1, 4)` | 12 | `[1,4,12]` |
| 4 | −12 +12 = 0 | `[4, 7)` | 12 | `[4,7,12]` |
| 7 | −12 | — (điểm cuối) | 0 | |

Hai đoạn riêng biệt dù tổng giống nhau. ✔

In [12]:
split_sweep_demo(examples[2][0])

[1, 4): cộng delta[1] =   12 -> tổng 12
[4, 7): cộng delta[4] =    0 -> tổng 12


[[1, 4, 12], [4, 7, 12]]

In [13]:
split_sweep_demo([[1, 3, 4], [5, 8, 6]])          # có khoảng trống [3, 5) không được sơn

[1, 3): cộng delta[1] =    4 -> tổng 4
[3, 5): cộng delta[3] =   -4 -> tổng 0  (không sơn, bỏ)
[5, 8): cộng delta[5] =    6 -> tổng 6


[[1, 3, 4], [5, 8, 6]]

## 9. Code nộp LeetCode

In [14]:
from collections import defaultdict
from typing import List

class Solution:
    def splitPainting(self, segments: List[List[int]]) -> List[List[int]]:
        delta = defaultdict(int)
        for start, end, color in segments:
            delta[start] += color        # màu vào
            delta[end] -= color          # màu ra (điểm vẫn được ghi nhận dù tổng delta = 0)
        points = sorted(delta)
        ans, cur = [], 0
        for i in range(len(points) - 1):
            cur += delta[points[i]]
            if cur:                      # cur > 0 <=> có ít nhất một màu (mọi màu >= 1)
                ans.append([points[i], points[i + 1], cur])
        return ans

for segs, want in examples:
    assert norm(Solution().splitPainting(segs)) == norm(want)
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `P` = tập mọi `start_i`, `end_i` (điểm biên), sắp xếp `p₀ < p₁ < … < p_k`.

**(1) Giữa hai điểm biên liên tiếp, tập màu không đổi.** Với `x ∈ [p_i, p_{i+1})`, đoạn `j` phủ `x` ⇔ `start_j ≤ x < end_j` ⇔ `start_j ≤ p_i` và `end_j ≥ p_{i+1}` (vì không có điểm biên nào trong `(p_i, p_{i+1})`). Điều kiện không phụ thuộc `x`.

**(2) Tổng tính đúng.** `cur` sau khi cộng `delta[p₀..p_i]` = `Σ_{start_j ≤ p_i} c_j − Σ_{end_j ≤ p_i} c_j` = tổng màu các đoạn đang phủ `[p_i, p_{i+1})`.

**(3) Tối thiểu.** Tại mỗi điểm biên `p_i` (với `0 < i < k`) có ít nhất một đoạn bắt đầu hoặc kết thúc ⇒ tập màu ngay trước và ngay sau `p_i` **khác nhau** (một màu cụ thể vào hoặc ra, và màu đôi một khác nhau nên không thể "bù trừ" ở mức tập hợp). Do đó mọi mô tả hợp lệ phải có một ranh giới tại mỗi `p_i` nằm giữa hai phần được sơn ⇒ không thể dùng ít đoạn hơn.

**(4) Bỏ phần không sơn:** khoảng có tập rỗng có `cur = 0` (màu dương) ⇒ bị bỏ, và ngược lại `cur > 0` ⇒ có màu.

## 11. Phân tích Time / Space

Gọi `n ≤ 2·10⁴` đoạn, `MAX = 10⁵`, `L = Σ (end − start)`.

| Cách | Time | Space |
|---|---|---|
| 1. Tô từng ô | `O(L)` (tới `2·10⁹`) | `O(MAX + L)` |
| 2. Mảng hiệu `MAX` | `O(n + MAX)` | `O(MAX)` |
| 3. Sự kiện thưa + sắp xếp | `O(n log n)` | `O(n)` |
| 4. accumulate + pairwise | `O(n log n)` | `O(n)` |

Tổng màu có thể tới `2·10⁴ · 10⁹ = 2·10¹³` — vượt `int32`/sát `int64`? (`2·10¹³ < 9.2·10¹⁸`) ⇒ `long` đủ; Python không lo.

In [15]:
rng = random.Random(1943)
colors = rng.sample(range(1, 10**9 + 1), 2 * 10**4)
segs = []
for c in colors:
    s = rng.randint(1, 10**5 - 1); e = rng.randint(s + 1, min(10**5, s + rng.choice([5, 500, 50000])))
    segs.append([s, e, c])
res = []
for name, f in [("Mảng hiệu", split_array), ("Sự kiện/Nộp", Solution().splitPainting), ("Gọn", SolutionCompact().splitPainting)]:
    t = time.perf_counter(); r = f(segs); res.append(tuple(map(tuple, norm(r))))
    print(f"{name}: {len(r)} đoạn trong {1000*(time.perf_counter()-t):.0f} ms")
assert len(set(res)) == 1

Mảng hiệu: 32844 đoạn trong 23 ms
Sự kiện/Nộp: 32844 đoạn trong 32 ms
Gọn: 32844 đoạn trong 34 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu tô từng ô)

In [16]:
sols = [Solution(), SolutionSweep(), SolutionCompact()]
edge = examples + [
    ([[1, 2, 3]], [[1, 2, 3]]),
    ([[1, 3, 4], [5, 8, 6]], [[1, 3, 4], [5, 8, 6]]),            # khoảng trống ở giữa
    ([[1, 5, 2], [2, 3, 3]], [[1, 2, 2], [2, 3, 5], [3, 5, 2]]),  # lồng nhau: hai đoạn ngoài cùng tổng 2 vẫn tách
    ([[1, 4, 10], [4, 8, 10 ** 9]], [[1, 4, 10], [4, 8, 10 ** 9]]),
    ([[3, 6, 1], [1, 3, 2], [6, 9, 3]], [[1, 3, 2], [3, 6, 1], [6, 9, 3]]),
]
for segs, want in edge:
    assert norm(split_brute(segs)) == norm(want), (segs, split_brute(segs))
    for s in sols:
        assert norm(s.splitPainting(segs)) == norm(want), (segs, s)
    assert norm(split_array(segs, 20 if max(e for _, e, _ in segs) < 20 else 10**5)) == norm(want)
    print("OK", segs, "->", norm(want))

OK [[1, 4, 5], [4, 7, 7], [1, 7, 9]] -> [[1, 4, 14], [4, 7, 16]]
OK [[1, 7, 9], [6, 8, 15], [8, 10, 7]] -> [[1, 6, 9], [6, 7, 24], [7, 8, 15], [8, 10, 7]]
OK [[1, 4, 5], [1, 4, 7], [4, 7, 1], [4, 7, 11]] -> [[1, 4, 12], [4, 7, 12]]
OK [[1, 2, 3]] -> [[1, 2, 3]]
OK [[1, 3, 4], [5, 8, 6]] -> [[1, 3, 4], [5, 8, 6]]
OK [[1, 5, 2], [2, 3, 3]] -> [[1, 2, 2], [2, 3, 5], [3, 5, 2]]
OK [[1, 4, 10], [4, 8, 1000000000]] -> [[1, 4, 10], [4, 8, 1000000000]]
OK [[3, 6, 1], [1, 3, 2], [6, 9, 3]] -> [[1, 3, 2], [3, 6, 1], [6, 9, 3]]


### 12.2 Ngẫu nhiên có seed: so với tô từng ô (tọa độ nhỏ)

In [17]:
rng = random.Random(42)
for _ in range(800):
    n = rng.randint(1, 8)
    cols = rng.sample(range(1, 50), n)
    segs = []
    for c in cols:
        s = rng.randint(1, 18); e = rng.randint(s + 1, 20)
        segs.append([s, e, c])
    want = norm(split_brute(segs))
    for sol in sols:
        assert norm(sol.splitPainting(segs)) == want, segs
    assert norm(split_array(segs, 25)) == want
print("800 bộ ngẫu nhiên: khớp tô từng ô")

800 bộ ngẫu nhiên: khớp tô từng ô


### 12.3 Kiểm tra tính chất của đầu ra

Các đoạn xuất ra không chồng nhau, phủ đúng phần được sơn, và hai đoạn **kề nhau** (chạm tại một điểm) luôn có **tập màu khác nhau** (tức không thể gộp).

In [18]:
def color_set(segs, x):
    return frozenset(c for s, e, c in segs if s <= x < e)

rng = random.Random(7)
for _ in range(300):
    cols = rng.sample(range(1, 40), rng.randint(1, 6))
    segs = [[s, rng.randint(s + 1, 15), c] for c in cols for s in [rng.randint(1, 13)]]
    out = sorted(Solution().splitPainting(segs))
    for (a, b, m), (c, d, _) in zip(out, out[1:]):
        assert b <= c                                         # không chồng
        if b == c:
            assert color_set(segs, a) != color_set(segs, c)   # kề nhau thì khác tập
    for a, b, m in out:
        assert all(color_set(segs, x) == color_set(segs, a) for x in range(a, b)) and sum(color_set(segs, a)) == m
print("đầu ra: không chồng, mỗi đoạn một tập màu, đoạn kề nhau luôn khác tập (tối thiểu)")

đầu ra: không chồng, mỗi đoạn một tập màu, đoạn kề nhau luôn khác tập (tối thiểu)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cắt theo "tổng thay đổi" (gộp các khoảng cùng tổng)

In [19]:
def wrong_merge_by_sum(segments):
    out = Solution().splitPainting(segments)
    out.sort()
    merged = []
    for a, b, m in out:
        if merged and merged[-1][1] == a and merged[-1][2] == m:     # SAI: cùng tổng chưa chắc cùng tập
            merged[-1][1] = b
        else:
            merged.append([a, b, m])
    return merged

segs, want = examples[2]
print("sai:", wrong_merge_by_sum(segs), "| đúng:", want)

sai: [[1, 7, 12]] | đúng: [[1, 4, 12], [4, 7, 12]]


**Vì sao sai:** đề nói rõ đoạn phải có cùng **tập** màu; `{5,7}` và `{1,11}` cùng tổng 12 nhưng khác tập. Chỉ nhìn tổng là mất thông tin. **Sửa:** cắt tại **mọi** điểm biên.

### Lỗi 2 — Bỏ các điểm có `delta = 0` (dùng `Counter` rồi lọc, hoặc xóa khóa)

In [20]:
def wrong_drop_zero(segments):
    delta = defaultdict(int)
    for s, e, c in segments:
        delta[s] += c; delta[e] -= c
    pts = sorted(p for p in delta if delta[p] != 0)            # SAI: mất điểm biên 4 ở Ví dụ 3
    out, cur = [], 0
    for a, b in pairwise(pts):
        cur += delta[a]
        if cur: out.append([a, b, cur])
    return out

segs, want = examples[2]
print("sai:", wrong_drop_zero(segs), "| đúng:", want)

sai: [[1, 7, 12]] | đúng: [[1, 4, 12], [4, 7, 12]]


**Vì sao sai:** điểm 4 có `delta = 0` (bốn màu vào/ra bù trừ về tổng), nhưng vẫn là nơi **tập màu đổi**. Lọc nó đi chính là gộp theo tổng. **Sửa:** giữ mọi điểm biên — `defaultdict` giữ khóa dù giá trị 0 (còn `Counter` thì các phép như `+c` sẽ tự xóa giá trị 0!).

### Lỗi 3 — Xuất cả khoảng không được sơn (`cur == 0`)

In [21]:
def wrong_keep_zero(segments):
    delta = defaultdict(int)
    for s, e, c in segments:
        delta[s] += c; delta[e] -= c
    pts = sorted(delta); out = []; cur = 0
    for a, b in pairwise(pts):
        cur += delta[a]
        out.append([a, b, cur])                                 # SAI: thiếu kiểm cur > 0
    return out

segs = [[1, 3, 4], [5, 8, 6]]
print("sai:", wrong_keep_zero(segs), "| đúng:", Solution().splitPainting(segs))

sai: [[1, 3, 4], [3, 5, 0], [5, 8, 6]] | đúng: [[1, 3, 4], [5, 8, 6]]


**Vì sao sai:** đề yêu cầu **bỏ** phần không được sơn; `[3, 5, 0]` không phải một đoạn màu.

### Lỗi 4 — Coi đoạn là **đóng** `[start, end]`

In [22]:
def wrong_closed(segments):
    delta = defaultdict(int)
    for s, e, c in segments:
        delta[s] += c; delta[e + 1] -= c                         # SAI: đoạn là nửa mở
    pts = sorted(delta); out = []; cur = 0
    for a, b in pairwise(pts):
        cur += delta[a]
        if cur: out.append([a, b, cur])
    return out

segs, want = examples[0]
print("sai:", wrong_closed(segs), "| đúng:", want)

sai: [[1, 4, 14], [4, 5, 21], [5, 8, 16]] | đúng: [[1, 4, 14], [4, 7, 16]]


**Vì sao sai:** `[1,4)` và `[4,7)` **không** chồng nhau tại 4. Trừ màu tại `end + 1` tạo ra vùng chồng giả `[4,5)`. **Sửa:** `delta[end] -= c`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Nhiều đoạn chồng nhau, hỏi giá trị gộp trên từng khoảng"** ⇒ **sweep line / mảng hiệu**: cộng ở đầu, trừ ở cuối, quét các điểm biên đã sắp xếp.
- Cắt theo **sự kiện**, không theo **giá trị** (giá trị có thể bằng nhau dù trạng thái khác).
- Đoạn **nửa mở** `[a, b)` ⇒ trừ tại `b`; đoạn đóng `[a, b]` ⇒ trừ tại `b + 1`.
- `defaultdict` giữ khóa có giá trị 0; `Counter` thì một số phép toán tự xóa — cẩn thận.
- Bài cùng mẫu: LeetCode 253 *Meeting Rooms II*, 731/732 *My Calendar*, 1094 *Car Pooling*, 2381 *Shifting Letters II*.

## 15. Biến thể của bài

1. **Xuất cả tập màu** (không chỉ tổng) ⇒ quét với `set` cập nhật tại điểm biên (Bài tập 3).
2. **Màu có thể trùng** ⇒ hai khoảng kề có thể cùng tập ⇒ phải so sánh tập (hoặc băm đa tập) để gộp.
3. **Hỏi khoảng có nhiều màu nhất / tổng lớn nhất** ⇒ cùng sweep, lấy max.
4. **Truy vấn trực tuyến** "thêm đoạn rồi hỏi màu tại điểm `x`" ⇒ BIT trên mảng hiệu.
5. **LeetCode 2158 — Amount of New Area Painted Each Day** ⇒ biến thể có thứ tự thời gian, dùng tập/segment tree.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tô từng ô | tập màu mỗi ô, gộp | `O(L)` | `O(MAX + L)` | kiểm chứng |
| 2. Mảng hiệu `MAX` | diff + cờ biên | `O(n + MAX)` | `O(MAX)` | tọa độ nhỏ |
| 3. Sự kiện + sắp xếp | dict delta + điểm biên | `O(n log n)` | `O(n)` | **nộp bài** |
| 4. accumulate + pairwise | như 3, viết gọn | `O(n log n)` | `O(n)` | code ngắn |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `[[1,3,1],[2,4,2]]` và `[[1,5,3],[1,5,4]]`?

**Bài 2 (tính tay):** Lập bảng điểm biên/`delta`/`cur` cho `[[1,5,2],[2,3,3]]`.

**Bài 3 (điền TODO):** Viết `split_with_sets(segments)` trả về `[left, right, sorted_list_of_colors]`.

**Bài 4 (tự viết lại):** Viết `max_mix(segments)` — tổng màu lớn nhất tại một điểm nào đó và khoảng `[l, r)` đầu tiên đạt giá trị đó.

**Bài 5 (mini challenge):** Biến thể: các màu **có thể trùng nhau** giữa các đoạn (màu trùng chỉ tính **một lần** trong tập trộn). Viết `split_dup_colors(segments)` cho ra mô tả **tối thiểu** — giờ hai khoảng kề có thể cùng tập màu, khi đó phải gộp.

In [23]:
def split_with_sets(segments):
    # TODO
    pass

def max_mix(segments):
    # TODO
    pass

def split_dup_colors(segments):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Điểm biên `1, 2, 3, 4` | `[1,2,1],[2,3,3],[3,4,2]` | Câu 2: `[1,5,7]` |
| 2 | `delta = {1:2, 2:3, 3:−3, 5:−2}` | `cur`: 2, 5, 2 | `[[1,2,2],[2,3,5],[3,5,2]]` |
| 3 | Lưu `adds[p]`, `removes[p]` | Duy trì `set` hiện tại | Xuất `sorted(set)` khi không rỗng |
| 4 | Quét như lời giải | Giữ max và khoảng đầu tiên | Trả `(max, [l, r])` |
| 5 | Màu trùng ⇒ đếm số đoạn đang phủ mỗi màu | Tập màu = các màu có đếm > 0 | Gộp khoảng kề nếu tập bằng nhau |

## 19. Đáp án tham khảo

In [24]:
r1 = [norm(Solution().splitPainting(s)) for s in ([[1, 3, 1], [2, 4, 2]], [[1, 5, 3], [1, 5, 4]])]
print("Bài 1:", r1); assert r1 == [[[1, 2, 1], [2, 3, 3], [3, 4, 2]], [[1, 5, 7]]]
print("Bài 2:"); assert split_sweep_demo([[1, 5, 2], [2, 3, 3]]) == [[1, 2, 2], [2, 3, 5], [3, 5, 2]]

Bài 1: [[[1, 2, 1], [2, 3, 3], [3, 4, 2]], [[1, 5, 7]]]
Bài 2:
[1, 2): cộng delta[1] =    2 -> tổng 2
[2, 3): cộng delta[2] =    3 -> tổng 5
[3, 5): cộng delta[3] =   -3 -> tổng 2


In [25]:
def split_with_sets(segments):
    adds, rems = defaultdict(list), defaultdict(list)
    for s, e, c in segments:
        adds[s].append(c); rems[e].append(c)
    pts = sorted(set(adds) | set(rems)); cur = set(); out = []
    for a, b in pairwise(pts):
        cur -= set(rems[a]); cur |= set(adds[a])
        if cur: out.append([a, b, sorted(cur)])
    return out

def max_mix(segments):
    out = sorted(Solution().splitPainting(segments))
    best = max(out, key=lambda t: t[2])                # max trả phần tử đầu tiên đạt max
    return best[2], best[:2]

rng = random.Random(3)
for _ in range(300):
    cols = rng.sample(range(1, 40), rng.randint(1, 6))
    segs = [[s, rng.randint(s + 1, 15), c] for c in cols for s in [rng.randint(1, 13)]]
    ws = split_with_sets(segs)
    assert [[a, b, sum(cs)] for a, b, cs in ws] == sorted(Solution().splitPainting(segs))
    assert all(set(cs) == color_set(segs, a) for a, b, cs in ws)
    v, (l, r) = max_mix(segs)
    assert v == max(sum(color_set(segs, x)) for x in range(1, 16))
print("Bài 3, 4: OK", split_with_sets(examples[2][0]), max_mix(examples[1][0]))

Bài 3, 4: OK [[1, 4, [5, 7]], [4, 7, [1, 11]]] (24, [6, 7])


In [26]:
def split_dup_colors(segments):
    events = defaultdict(list)
    for s, e, c in segments:
        events[s].append((c, 1)); events[e].append((c, -1))
    pts = sorted(events); cnt = defaultdict(int); out = []
    for a, b in pairwise(pts):
        for c, d in events[a]:
            cnt[c] += d
        cur = frozenset(c for c, k in cnt.items() if k > 0)
        if not cur:
            continue
        if out and out[-1][1] == a and out[-1][3] == cur:
            out[-1][1] = b                                  # cùng tập màu: gộp
        else:
            out.append([a, b, sum(cur), cur])
    return [[a, b, m] for a, b, m, _ in out]

def dup_brute(segments):
    hi = max(e for _, e, _ in segments)
    cover = [frozenset(c for s, e, c in segments if s <= x < e) for x in range(hi + 1)]
    out = []; x = 1
    while x < hi:
        if not cover[x]: x += 1; continue
        y = x
        while y < hi and cover[y] == cover[x]: y += 1
        out.append([x, y, sum(cover[x])]); x = y
    return out

rng = random.Random(9)
for _ in range(500):
    segs = [[s, rng.randint(s + 1, 15), rng.randint(1, 4)] for s in (rng.randint(1, 13) for _ in range(rng.randint(1, 6)))]
    assert split_dup_colors(segs) == dup_brute(segs), segs
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", split_dup_colors([[1, 4, 5], [4, 7, 5]]))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK [[1, 7, 5]]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
delta = defaultdict(int)
for s, e, c in segments:
    delta[s] += c; delta[e] -= c          # nửa mở: trừ tại e
pts = sorted(delta)                         # GIỮ cả điểm có delta = 0
ans, cur = [], 0
for a, b in zip(pts, pts[1:]):
    cur += delta[a]
    if cur: ans.append([a, b, cur])
return ans
```

**Checklist:**
- [ ] Cắt tại **mọi** điểm biên (kể cả `delta = 0`), không gộp theo tổng?
- [ ] Đoạn nửa mở: `delta[end] -= c` (không phải `end + 1`)?
- [ ] Bỏ khoảng `cur == 0` (không được sơn)?
- [ ] Sắp xếp điểm biên trước khi quét?
- [ ] Không dùng `Counter` theo cách tự xóa khóa 0.